# Phase 1: Exploratory Data Analysis (EDA)
## ML Challenge 2025 - Smart Product Pricing

**Objective**: Understand the training data structure, distributions, and patterns

**Tasks**:
1. Load and inspect training data
2. Analyze price distribution
3. Explore catalog_content text features
4. Check image_link availability
5. Identify missing values and outliers

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
warnings.filterwarnings('ignore')
%matplotlib inline

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

print("✅ Libraries imported successfully!")

## Task 1.1: Load and Explore Training Data

In [ ]:
# Load training data
train_df = pd.read_csv('../dataset/train.csv')

print(f"📊 Training Data Shape: {train_df.shape}")
print(f"   Rows: {train_df.shape[0]:,}")
print(f"   Columns: {train_df.shape[1]}")
print("\n" + "="*80 + "\n")

# Display first few rows
print("📋 First 5 rows of training data:")
train_df.head()

In [ ]:
# Dataset information
print("📊 Dataset Information:")
print("="*80)
train_df.info()

In [ ]:
# Check for missing values
print("🔍 Missing Values Analysis:")
print("="*80)
missing = train_df.isnull().sum()
missing_pct = (missing / len(train_df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Percentage': missing_pct
})
print(missing_df)
print(f"\n✅ Total missing values: {missing.sum():,}")

In [ ]:
# Basic statistics
print("📈 Statistical Summary:")
print("="*80)
train_df.describe()

## Task 1.2: Price Distribution Analysis

In [ ]:
# Price statistics
print("💰 Price Statistics:")
print("="*80)
print(f"Mean Price:     ${train_df['price'].mean():,.2f}")
print(f"Median Price:   ${train_df['price'].median():,.2f}")
print(f"Std Dev:        ${train_df['price'].std():,.2f}")
print(f"Min Price:      ${train_df['price'].min():,.2f}")
print(f"Max Price:      ${train_df['price'].max():,.2f}")
print(f"\nPrice Range:    ${train_df['price'].min():,.2f} - ${train_df['price'].max():,.2f}")
print(f"\nQuartiles:")
print(f"  25th percentile: ${train_df['price'].quantile(0.25):,.2f}")
print(f"  50th percentile: ${train_df['price'].quantile(0.50):,.2f}")
print(f"  75th percentile: ${train_df['price'].quantile(0.75):,.2f}")
print(f"  95th percentile: ${train_df['price'].quantile(0.95):,.2f}")
print(f"  99th percentile: ${train_df['price'].quantile(0.99):,.2f}")

In [ ]:
# Visualize price distribution
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Histogram of prices
axes[0, 0].hist(train_df['price'], bins=100, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Price Distribution', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Price ($)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].axvline(train_df['price'].mean(), color='red', linestyle='--', label=f'Mean: ${train_df["price"].mean():.2f}')
axes[0, 0].axvline(train_df['price'].median(), color='green', linestyle='--', label=f'Median: ${train_df["price"].median():.2f}')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Log-transformed price distribution
axes[0, 1].hist(np.log1p(train_df['price']), bins=100, edgecolor='black', alpha=0.7, color='orange')
axes[0, 1].set_title('Log-Transformed Price Distribution', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Log(Price + 1)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].grid(True, alpha=0.3)

# 3. Box plot
axes[1, 0].boxplot(train_df['price'], vert=True)
axes[1, 0].set_title('Price Box Plot (Outliers Visible)', fontsize=14, fontweight='bold')
axes[1, 0].set_ylabel('Price ($)')
axes[1, 0].grid(True, alpha=0.3)

# 4. Price distribution (zoomed in - up to 95th percentile)
price_95 = train_df['price'].quantile(0.95)
axes[1, 1].hist(train_df[train_df['price'] <= price_95]['price'], bins=100, edgecolor='black', alpha=0.7, color='green')
axes[1, 1].set_title('Price Distribution (up to 95th percentile)', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Price ($)')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ Price distribution visualizations created!")

In [ ]:
# Identify price outliers using IQR method
Q1 = train_df['price'].quantile(0.25)
Q3 = train_df['price'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers_low = train_df[train_df['price'] < lower_bound]
outliers_high = train_df[train_df['price'] > upper_bound]

print("🔍 Outlier Analysis (IQR Method):")
print("="*80)
print(f"IQR: ${IQR:.2f}")
print(f"Lower Bound: ${lower_bound:.2f}")
print(f"Upper Bound: ${upper_bound:.2f}")
print(f"\nLow Outliers (< ${lower_bound:.2f}): {len(outliers_low):,} ({len(outliers_low)/len(train_df)*100:.2f}%)")
print(f"High Outliers (> ${upper_bound:.2f}): {len(outliers_high):,} ({len(outliers_high)/len(train_df)*100:.2f}%)")
print(f"\nTotal Outliers: {len(outliers_low) + len(outliers_high):,} ({(len(outliers_low) + len(outliers_high))/len(train_df)*100:.2f}%)")

## Task 1.3: Text Analysis (catalog_content)

In [ ]:
# Sample catalog content
print("📝 Sample Catalog Content:")
print("="*80)
for i in range(5):
    print(f"\n[Sample {i+1}]")
    print(f"ID: {train_df.iloc[i]['sample_id']}")
    print(f"Price: ${train_df.iloc[i]['price']:.2f}")
    print(f"Content: {train_df.iloc[i]['catalog_content'][:200]}...")
    print("-"*80)

In [ ]:
# Text statistics
train_df['text_length'] = train_df['catalog_content'].str.len()
train_df['word_count'] = train_df['catalog_content'].str.split().str.len()
train_df['char_count'] = train_df['catalog_content'].str.len()

print("📊 Text Statistics:")
print("="*80)
print(f"\nText Length (characters):")
print(f"  Mean: {train_df['text_length'].mean():.2f}")
print(f"  Median: {train_df['text_length'].median():.2f}")
print(f"  Min: {train_df['text_length'].min()}")
print(f"  Max: {train_df['text_length'].max()}")

print(f"\nWord Count:")
print(f"  Mean: {train_df['word_count'].mean():.2f}")
print(f"  Median: {train_df['word_count'].median():.2f}")
print(f"  Min: {train_df['word_count'].min()}")
print(f"  Max: {train_df['word_count'].max()}")

In [ ]:
# Visualize text statistics
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Text length distribution
axes[0].hist(train_df['text_length'], bins=100, edgecolor='black', alpha=0.7)
axes[0].set_title('Text Length Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Character Count')
axes[0].set_ylabel('Frequency')
axes[0].axvline(train_df['text_length'].mean(), color='red', linestyle='--', label=f'Mean: {train_df["text_length"].mean():.0f}')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Word count distribution
axes[1].hist(train_df['word_count'], bins=100, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_title('Word Count Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Frequency')
axes[1].axvline(train_df['word_count'].mean(), color='red', linestyle='--', label=f'Mean: {train_df["word_count"].mean():.0f}')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ Text statistics visualizations created!")

In [ ]:
# Correlation between text length and price
correlation = train_df[['text_length', 'word_count', 'price']].corr()

print("🔗 Correlation with Price:")
print("="*80)
print(correlation['price'].sort_values(ascending=False))

# Visualize correlation
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Scatter: text length vs price
axes[0].scatter(train_df['text_length'], train_df['price'], alpha=0.3, s=10)
axes[0].set_title('Text Length vs Price', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Text Length (characters)')
axes[0].set_ylabel('Price ($)')
axes[0].grid(True, alpha=0.3)

# Scatter: word count vs price
axes[1].scatter(train_df['word_count'], train_df['price'], alpha=0.3, s=10, color='orange')
axes[1].set_title('Word Count vs Price', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Price ($)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Extract potential patterns - look for numbers (IPQ, measurements, etc.)
import re

def extract_numbers(text):
    """Extract all numbers from text"""
    return re.findall(r'\d+', str(text))

train_df['numbers'] = train_df['catalog_content'].apply(extract_numbers)
train_df['num_count'] = train_df['numbers'].apply(len)

print("🔢 Numeric Patterns in Text:")
print("="*80)
print(f"Average numbers per product: {train_df['num_count'].mean():.2f}")
print(f"Max numbers in a product: {train_df['num_count'].max()}")
print(f"\nSamples with numbers:")
print(train_df[train_df['num_count'] > 0][['catalog_content', 'numbers', 'price']].head())

## Task 1.4: Image Link Analysis

In [ ]:
# Analyze image links
print("🖼️ Image Link Analysis:")
print("="*80)

print(f"\nTotal samples: {len(train_df):,}")
print(f"Samples with image links: {train_df['image_link'].notna().sum():,}")
print(f"Samples without image links: {train_df['image_link'].isna().sum():,}")
print(f"\nPercentage with images: {train_df['image_link'].notna().sum() / len(train_df) * 100:.2f}%")

print("\n📝 Sample image links:")
for i, link in enumerate(train_df['image_link'].dropna().head(5)):
    print(f"{i+1}. {link}")

In [ ]:
# Check image link patterns
train_df['has_image'] = train_df['image_link'].notna()

# Price comparison: with vs without images
price_with_image = train_df[train_df['has_image']]['price']
price_without_image = train_df[~train_df['has_image']]['price']

print("💰 Price Comparison (With vs Without Images):")
print("="*80)
print(f"\nWith Images:")
print(f"  Count: {len(price_with_image):,}")
print(f"  Mean Price: ${price_with_image.mean():.2f}")
print(f"  Median Price: ${price_with_image.median():.2f}")

if len(price_without_image) > 0:
    print(f"\nWithout Images:")
    print(f"  Count: {len(price_without_image):,}")
    print(f"  Mean Price: ${price_without_image.mean():.2f}")
    print(f"  Median Price: ${price_without_image.median():.2f}")
else:
    print(f"\n✅ All samples have image links!")

## Task 1.5: Test Data Exploration

In [ ]:
# Load test data
test_df = pd.read_csv('../dataset/test.csv')

print("📊 Test Data Shape:")
print(f"   Rows: {test_df.shape[0]:,}")
print(f"   Columns: {test_df.shape[1]}")
print("\n" + "="*80 + "\n")

print("📋 First 5 rows of test data:")
test_df.head()

In [ ]:
# Test data info
print("📊 Test Dataset Information:")
print("="*80)
test_df.info()

print("\n🔍 Missing Values in Test Data:")
print("="*80)
missing_test = test_df.isnull().sum()
missing_test_pct = (missing_test / len(test_df)) * 100
missing_test_df = pd.DataFrame({
    'Missing Count': missing_test,
    'Percentage': missing_test_pct
})
print(missing_test_df)

In [ ]:
# Compare train and test distributions
test_df['text_length'] = test_df['catalog_content'].str.len()
test_df['word_count'] = test_df['catalog_content'].str.split().str.len()

print("📊 Train vs Test Comparison:")
print("="*80)
print(f"\nText Length:")
print(f"  Train Mean: {train_df['text_length'].mean():.2f}")
print(f"  Test Mean:  {test_df['text_length'].mean():.2f}")

print(f"\nWord Count:")
print(f"  Train Mean: {train_df['word_count'].mean():.2f}")
print(f"  Test Mean:  {test_df['word_count'].mean():.2f}")

print(f"\nImage Availability:")
print(f"  Train: {train_df['image_link'].notna().sum() / len(train_df) * 100:.2f}%")
print(f"  Test:  {test_df['image_link'].notna().sum() / len(test_df) * 100:.2f}%")

In [ ]:
# Visualize train vs test distributions
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Text length comparison
axes[0].hist(train_df['text_length'], bins=50, alpha=0.5, label='Train', edgecolor='black')
axes[0].hist(test_df['text_length'], bins=50, alpha=0.5, label='Test', edgecolor='black')
axes[0].set_title('Text Length Distribution: Train vs Test', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Text Length (characters)')
axes[0].set_ylabel('Frequency')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Word count comparison
axes[1].hist(train_df['word_count'], bins=50, alpha=0.5, label='Train', edgecolor='black')
axes[1].hist(test_df['word_count'], bins=50, alpha=0.5, label='Test', edgecolor='black')
axes[1].set_title('Word Count Distribution: Train vs Test', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Frequency')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ Train vs Test comparison visualizations created!")

## Summary of EDA Findings

In [ ]:
print("\n" + "="*80)
print("📊 PHASE 1 SUMMARY: KEY FINDINGS")
print("="*80)

print("\n1️⃣ DATASET SIZE:")
print(f"   ✓ Training samples: {len(train_df):,}")
print(f"   ✓ Test samples: {len(test_df):,}")

print("\n2️⃣ PRICE DISTRIBUTION:")
print(f"   ✓ Price range: ${train_df['price'].min():.2f} - ${train_df['price'].max():.2f}")
print(f"   ✓ Mean price: ${train_df['price'].mean():.2f}")
print(f"   ✓ Median price: ${train_df['price'].median():.2f}")
print(f"   ✓ Outliers: {(len(outliers_low) + len(outliers_high))/len(train_df)*100:.2f}% of data")

print("\n3️⃣ TEXT FEATURES:")
print(f"   ✓ Average text length: {train_df['text_length'].mean():.0f} characters")
print(f"   ✓ Average word count: {train_df['word_count'].mean():.0f} words")
print(f"   ✓ Text-price correlation: {train_df[['text_length', 'price']].corr().iloc[0, 1]:.3f}")

print("\n4️⃣ IMAGE AVAILABILITY:")
print(f"   ✓ Train images: {train_df['image_link'].notna().sum() / len(train_df) * 100:.2f}%")
print(f"   ✓ Test images: {test_df['image_link'].notna().sum() / len(test_df) * 100:.2f}%")

print("\n5️⃣ MISSING VALUES:")
print(f"   ✓ Training data: {train_df.isnull().sum().sum()} missing values")
print(f"   ✓ Test data: {test_df.isnull().sum().sum()} missing values")

print("\n6️⃣ RECOMMENDATIONS FOR NEXT PHASE:")
print("   ✓ Consider log-transformation for price (right-skewed distribution)")
print("   ✓ Extract Item Pack Quantity (IPQ) from catalog_content")
print("   ✓ Extract brand names and product categories")
print("   ✓ Download and process product images")
print("   ✓ Handle price outliers carefully (cap or transform)")
print("   ✓ Create text embeddings for better feature representation")

print("\n" + "="*80)
print("✅ PHASE 1 COMPLETE: Ready for Phase 2 (Feature Engineering)")
print("="*80)

## Next Steps

Based on this EDA, proceed to **Phase 2: Feature Engineering** to:
1. Extract IPQ (Item Pack Quantity) from text
2. Create advanced text features and embeddings
3. Download and extract image features
4. Prepare features for modeling